In [26]:
from dotenv import load_dotenv
import anthropic, json, pandas as pd
from pathlib import Path

In [27]:
load_dotenv()
client = anthropic.Anthropic()  # lee ANTHROPIC_API_KEY del entorno

In [33]:
system_md = Path("../classifier/campo_gravitatorio.md").read_text(encoding="utf-8")
df = pd.read_csv("../classifier/sample_classification.csv")

In [29]:
def classify(enunciado, system_md, modelo="claude-haiku-4-5-20251001"):
    resp = client.messages.create(
        model=modelo,
        max_tokens=300,
        system=[{"type": "text", "text": system_md,
                 "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": f"Enunciado:\n{enunciado}"}],
    )
    texto = resp.content[0].text.strip().removeprefix("```json").removesuffix("```").strip()
    return json.loads(texto)

In [51]:
output_file = Path("../classifier/resultados.jsonl")
already_processed = {json.loads(l)["id"] for l in output_file.open()} if output_file.exists() else set()

In [52]:
already_processed

set()

In [53]:
for index, row in df.iterrows():
    if row.id in already_processed:
        continue
    try:
        r = classify(row.enunciado, system_md)
        r["id"] = row.id
    except Exception as e:
        r = {"id": row.id, "error": str(e)}
    with output_file.open("a", encoding="utf-8") as f:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [48]:
df

,id,año,convocatoria,ejercicio,opcion,enunciado,imagen,aviso,asignatura
0,2022_Junio_A1,2022,Junio,A1,NaN,a) i) Defina los conceptos de energía cinética...,enunciados_PNG/2022 - Campo gravitatorio_png/2...,NaN,Física
1,2024_Julio_A2,2024,Julio,A2,NaN,a) Razone si son verdaderos los siguientes enu...,enunciados_PNG/2024 - Campo gravitatorio_png/2...,NaN,Física
2,2024_Reserva 2_A2,2024,Reserva 2,A2,NaN,a) i) Enuncie la tercera ley de Kepler identif...,enunciados_PNG/2024 - Campo gravitatorio_png/2...,NaN,Física
3,2023_Junio_A2,2023,Junio,A2,NaN,a) i) Escriba la expresión del potencial gravi...,enunciados_PNG/2023 - Campo gravitatorio_png/2...,NaN,Física
4,2016_Reserva 2_1_A,2016,Reserva 2,1,A,a) Enuncie las leyes de Kepler.\nb) Dos satéli...,enunciados_PNG/2016 - Campo gravitatorio_png/2...,NaN,Física
5,2024_Reserva 1_A2,2024,Reserva 1,A2,NaN,a) Sobre una partícula que describe una trayec...,enunciados_PNG/2024 - Campo gravitatorio_png/2...,NaN,Física
6,2021_Julio_A1,2021,Julio,A1,NaN,a) Razone si la siguiente afirmación es verdad...,enunciados_PNG/2021 - Campo gravitatorio_png/2...,NaN,Física
7,2020_Reserva 4_1,2020,Reserva 4,1,NaN,a) Un planeta A tiene triple de masa y doble d...,enunciados_PNG/2020 - Campo gravitatorio_png/2...,NaN,Física
8,2017_Junio_1_B,2017,Junio,1,B,a) Un bloque de acero está situado sobre la su...,enunciados_PNG/2017 - Campo gravitatorio_png/2...,NaN,Física
9,2021_Reserva 3_A2,2021,Reserva 3,A2,NaN,"a) Dos satélites idénticos, A y B, están en ór...",enunciados_PNG/2021 - Campo gravitatorio_png/2...,NaN,Física


In [54]:
df_out = (pd.read_json(output_file, lines=True)
          .sort_values("id")
)
df_out = df_out[["id"] + list(df_out.columns[:-2])]

df_out


,id,asignatura,tema,tipo_ejercicio
4,2016_Reserva 2_1_A,Física,Campo gravitatorio,satélites
8,2017_Junio_1_B,Física,Campo gravitatorio,satélites
7,2020_Reserva 4_1,Física,Campo gravitatorio,distribución_masas
6,2021_Julio_A1,Física,Campo gravitatorio,satélites
9,2021_Reserva 3_A2,Física,Campo gravitatorio,satélites
0,2022_Junio_A1,Física,Campo gravitatorio,plano_inclinado
3,2023_Junio_A2,Física,Campo gravitatorio,distribución_masas
1,2024_Julio_A2,Física,Campo gravitatorio,plano_inclinado
5,2024_Reserva 1_A2,Física,Campo gravitatorio,plano_inclinado
2,2024_Reserva 2_A2,Física,Campo gravitatorio,satélites
